[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/14_model_evaluation/03_regression_metrics.ipynb)

# Bài 03 — Metric cho hồi quy

**🎯 Mục tiêu**
- MAE, MSE, RMSE, $R^2$, MAPE, sMAPE, WAPE, RMSLE, MedAE — ý nghĩa, đơn vị, độ nhạy với outlier.
- Metric nào được tối ưu bởi dự đoán nào: **trung bình** (MSE) vs **trung vị** (MAE) vs **phân vị** (pinball loss).
- Bẫy: MAPE với giá trị gần 0, $R^2$ âm, so sánh $R^2$ giữa các tập dữ liệu.
- Luôn so với **baseline** và nhìn **phân phối sai số**, không chỉ một con số.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/14_model_evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from ml_course import metrics as mm
from ml_course.visualization import plot_residuals, set_style

set_style()
houses = pd.read_csv("../../data/sample/house_prices.csv")
X = pd.get_dummies(houses.drop(columns=["price_billion_vnd"]), columns=["district"], dtype=float)
y = houses["price_billion_vnd"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
preds = {"baseline (trung bình train)": np.full(len(y_te), y_tr.mean()),
         "Linear Regression": LinearRegression().fit(X_tr, y_tr).predict(X_te),
         "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0).fit(X_tr, y_tr).predict(X_te)}

## 1. Bảng metric

| Metric | Công thức | Đơn vị | Ghi chú |
|---|---|---|---|
| MAE | $\frac1n\sum\lvert y - \hat y\rvert$ | như $y$ | dễ hiểu; robust hơn với outlier |
| MSE | $\frac1n\sum(y - \hat y)^2$ | $y^2$ | phạt nặng sai số lớn; khả vi |
| RMSE | $\sqrt{\text{MSE}}$ | như $y$ | ≥ MAE; chênh lệch lớn → có vài sai số rất lớn |
| $R^2$ | $1 - \frac{\sum(y - \hat y)^2}{\sum(y - \bar y)^2}$ | — | so với baseline "đoán trung bình"; có thể **âm** |
| MAPE | $\frac{100}{n}\sum\left\lvert\frac{y - \hat y}{y}\right\rvert$ | % | không xác định khi $y = 0$; bất đối xứng |
| WAPE | $\frac{\sum\lvert y - \hat y\rvert}{\sum\lvert y\rvert}$ | % | ổn định hơn MAPE (dự báo nhu cầu) |
| RMSLE | $\sqrt{\frac1n\sum(\log(1 + \hat y) - \log(1 + y))^2}$ | — | sai số **tương đối**; phạt dự đoán thấp nặng hơn |
| MedAE | trung vị $\lvert y - \hat y\rvert$ | như $y$ | rất robust |

In [ ]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))


rows = []
for name, p in preds.items():
    rows.append({"model": name, "MAE": mm.mean_absolute_error(y_te, p), "RMSE": mm.root_mean_squared_error(y_te, p),
                 "R²": mm.r2_score(y_te, p), "MAPE %": 100 * mm.mean_absolute_percentage_error(y_te, p),
                 "WAPE %": 100 * wape(y_te, p), "RMSLE": np.sqrt(skm.mean_squared_log_error(y_te, np.maximum(p, 0))),
                 "MedAE": skm.median_absolute_error(y_te, p)})
pd.DataFrame(rows).set_index("model").round(3)

Baseline có $R^2 \approx 0$ (theo định nghĩa) — mọi model phải vượt nó. Đơn vị của MAE/RMSE là **tỉ VND** → dễ giải thích cho người dùng ("sai trung bình ~0.5 tỉ").

Để ý: HistGradientBoosting thắng ở MAE, MAPE, MedAE nhưng **thua** Linear Regression ở RMSE/R² — nó sai ít hơn với đa số căn nhà nhưng có vài sai số rất lớn. **Chọn metric khác → xếp hạng model khác** — vì vậy phải chọn metric **trước** khi so sánh.

## 2. MSE tối ưu bởi trung bình, MAE bởi trung vị

Nếu chỉ được đoán **một hằng số** $c$ cho mọi mẫu: $\arg\min_c \sum(y - c)^2 = \bar y$, còn $\arg\min_c\sum\lvert y - c\rvert = \operatorname{median}(y)$. Vì vậy metric bạn chọn ngầm quyết định model nên ước lượng **đại lượng thống kê** nào:

In [ ]:
rng = np.random.default_rng(0)
incomes = rng.lognormal(np.log(15), 0.8, 2000)                     # thu nhập (triệu) — phân phối lệch phải
cs = np.linspace(5, 60, 600)
mse = [np.mean((incomes - c) ** 2) for c in cs]
mae = [np.mean(np.abs(incomes - c)) for c in cs]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(cs, mse)
axes[0].axvline(incomes.mean(), color="r", ls="--", label=f"trung bình = {incomes.mean():.1f}")
axes[0].set_title("MSE của dự đoán hằng số c")
axes[1].plot(cs, mae)
axes[1].axvline(np.median(incomes), color="g", ls="--", label=f"trung vị = {np.median(incomes):.1f}")
axes[1].set_title("MAE của dự đoán hằng số c")
for ax in axes:
    ax.set_xlabel("c")
    ax.legend()
plt.show()

**Pinball (quantile) loss** tổng quát hoá: tối thiểu bởi phân vị $\tau$ — dùng cho dự báo khoảng (vd "doanh số ở phân vị 90% để đặt hàng tồn kho"):

$$L_\tau(y, \hat y) = \max\big(\tau(y - \hat y),\ (\tau - 1)(y - \hat y)\big)$$

In [ ]:
for tau in [0.1, 0.5, 0.9]:
    pinball = [np.mean(np.maximum(tau * (incomes - c), (tau - 1) * (incomes - c))) for c in cs]
    print(f"τ = {tau}: c tối ưu = {cs[int(np.argmin(pinball))]:.2f} | phân vị {tau} thực tế = {np.quantile(incomes, tau):.2f}")

## 3. Ảnh hưởng của outlier

In [ ]:
p = preds["HistGradientBoosting"].copy()
p_bad = p.copy()
p_bad[:3] += 15                                         # 3 dự đoán sai rất lớn (vd lỗi dữ liệu)
for name, pp in [("gốc", p), ("thêm 3 sai số cực lớn", p_bad)]:
    print(f"{name:22s} MAE = {mm.mean_absolute_error(y_te, pp):.3f} | RMSE = {mm.root_mean_squared_error(y_te, pp):.3f} | "
          f"MedAE = {skm.median_absolute_error(y_te, pp):.3f}")

RMSE tăng mạnh hơn nhiều so với MAE; MedAE gần như không đổi. Chọn metric theo việc **sai số lớn** có đặc biệt nghiêm trọng không (vd dự báo phụ tải điện: có → RMSE).

## 4. Bẫy của MAPE và R²

In [ ]:
y_small = np.array([0.01, 0.5, 10, 20, 30.0])
p_small = np.array([0.11, 0.6, 10.1, 20.1, 30.1])                 # sai đúng 0.1 ở mọi điểm
print(f"sai số tuyệt đối mọi điểm = 0.1 nhưng MAPE = {100 * mm.mean_absolute_percentage_error(y_small, p_small):.0f}% "
      f"(điểm y = 0.01 đóng góp 1000%) | WAPE = {100 * wape(y_small, p_small):.2f}%")

y_r = rng.normal(0, 1, 200)
print(f"\nR² của dự đoán tệ hơn trung bình (đảo dấu): {mm.r2_score(y_r, -y_r):.2f}  → R² có thể âm")
x_wide = rng.uniform(0, 10, 500)
x_narrow = x_wide[x_wide < 2]
for name, xx in [("x ∈ [0, 10]", x_wide), ("x ∈ [0, 2]", x_narrow)]:
    yy = 2 * xx + rng.normal(0, 1, len(xx))
    print(f"{name}: cùng quy luật y = 2x + nhiễu(σ=1) → R² = {mm.r2_score(yy, 2 * xx):.3f} | RMSE = {mm.root_mean_squared_error(yy, 2 * xx):.3f}")

- MAPE bùng nổ khi $y$ gần 0 (doanh số ngày thấp, sản phẩm mới) → dùng WAPE hoặc MAE.
- Cùng **một** model đúng (sai số σ = 1 như nhau), $R^2$ khác hẳn chỉ vì **độ phân tán của y** trong tập đánh giá → **không so sánh R² giữa các tập dữ liệu khác nhau**; RMSE thì ổn định.

## 5. Nhìn phân phối sai số

In [ ]:
plot_residuals(y_te, preds["HistGradientBoosting"], title="HistGradientBoosting — giá nhà")
plt.show()
abs_err = np.abs(y_te - preds["HistGradientBoosting"])
print("phân vị sai số tuyệt đối (tỉ):", dict(zip(["50%", "80%", "95%", "99%"], np.quantile(abs_err, [0.5, 0.8, 0.95, 0.99]).round(3))))
seg = pd.DataFrame({"giá thật": pd.qcut(y_te, 4, labels=["rẻ nhất", "Q2", "Q3", "đắt nhất"]), "sai số": abs_err})
seg.groupby("giá thật", observed=True)["sai số"].agg(["mean", "median"]).round(3)

Một con số MAE che giấu chuyện model sai **nhiều hơn hẳn** ở phân khúc nhà đắt — báo cáo theo phân khúc và phân vị sai số giúp người dùng biết khi nào tin model.

## 🧠 Kiểm tra nhanh

1. Model A: MAE 1.0, RMSE 1.1. Model B: MAE 0.9, RMSE 2.5. Nói gì về phân phối sai số của hai model?
2. Dự báo tồn kho, thiếu hàng tốn kém gấp 9 lần thừa hàng. Nên dự báo đại lượng nào? Metric nào?
3. Vì sao không nên dùng MAPE cho doanh số theo ngày của sản phẩm bán chậm?
4. $R^2 = -0.3$ nghĩa là gì?

## 📌 Tóm tắt

- MAE ↔ trung vị, MSE ↔ trung bình, pinball ↔ phân vị: chọn metric = chọn đại lượng cần dự đoán.
- RMSE nhạy với sai số lớn; MAPE hỏng khi y gần 0 (dùng WAPE); R² phụ thuộc phương sai của y.
- Luôn so với baseline và xem phân phối sai số theo phân khúc.

## 📚 Tài liệu tham khảo

- Hyndman & Koehler (2006) — *Another look at measures of forecast accuracy*
- Gneiting (2011) — *Making and Evaluating Point Forecasts*
- scikit-learn User Guide — Regression metrics